# FoodGuard AI — Food Safety Reasoning Agent (LLM)

**Purpose**: Orchestrate the FoodSafetyAgent using LangGraph + Ollama Qwen3.

This agent:
1. Reads correlation agent output from DB
2. Gets all 3 predictions (aroma/taste/vision + confidence)
3. Calls Ollama Qwen3 via structured prompt
4. Parses response to extract risk level & reasoning
5. Updates investigation with final verdict
6. Logs execution to agent_execution table

---

**Requirements**: Ollama must be running locally (http://localhost:11434)

**Run this AFTER notebook 05 (correlation engine).**

**Can run standalone** for testing.

## 1. Imports & Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import time
from typing import Dict, Any, Optional
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END

from foodguard_lib import (
    DB_PATH,
    get_aroma_analysis,
    get_taste_analysis,
    get_vision_analysis,
    get_investigation,
    execute_query,
    insert_agent_execution,
    update_investigation,
    get_ollama_client,
    dict_from_row
)

print("[OK] Imports successful")

## 2. Check Ollama Availability

In [ ]:
# Check if Ollama is running
ollama_client = get_ollama_client()
is_available = ollama_client.is_available()

if is_available:
    print("[OK] Ollama is running and accessible")
else:
    print("[WARNING] Ollama not accessible. Will use fallback mock responses.")
    print("To use real LLM, start Ollama: ollama serve")

## 3. Define Agent State

In [ ]:
class FoodSafetyAgentState(TypedDict):
    """State dictionary for FoodSafetyAgent."""
    # Inputs
    investigation_id: str
    batch_id: str
    suspected_adulterant: str
    correlation_confidence: float
    aroma_pred: Dict[str, Any]  # {"class": str, "confidence": float}
    taste_pred: Dict[str, Any]
    vision_pred: Dict[str, Any]
    matched_rules: int  # count

    # Outputs
    llm_response: str
    risk_level: str  # "Low", "Medium", "High", "Critical"
    reasoning: str
    final_verdict: str

    # Tracking
    execution_id: Optional[str]
    execution_time_ms: float

print("[OK] FoodSafetyAgentState defined")

## 4. Define FoodSafetyAgent Class

In [ ]:
class FoodSafetyAgent:
    """
    LangGraph-based food safety agent.
    Calls LLM for reasoning + risk assessment.
    """

    def __init__(self, db_path: str = DB_PATH, ollama_client=None):
        self.db_path = db_path
        self.ollama_client = ollama_client or get_ollama_client()
        self.graph = StateGraph(FoodSafetyAgentState)
        self._build_graph()
        self.compiled_graph = self.graph.compile()

    def _build_graph(self):
        """Build the LangGraph workflow."""
        # Add nodes
        self.graph.add_node("build_prompt", self.build_prompt)
        self.graph.add_node("call_llm", self.call_llm)
        self.graph.add_node("parse_response", self.parse_response)
        self.graph.add_node("finalize", self.finalize)

        # Add edges
        self.graph.add_edge("build_prompt", "call_llm")
        self.graph.add_edge("call_llm", "parse_response")
        self.graph.add_edge("parse_response", "finalize")
        self.graph.add_edge("finalize", END)

        # Set entry point
        self.graph.set_entry_point("build_prompt")

    def build_prompt(self, state: FoodSafetyAgentState) -> FoodSafetyAgentState:
        """
        Build the LLM prompt with all relevant context.
        """
        prompt = f"""You are a food safety investigator analyzing a milk sample.

**Investigation ID**: {state['investigation_id']}

**Sensor Predictions**:
- **Aroma (E-Nose)**: {state['aroma_pred']['class']} (confidence: {state['aroma_pred']['confidence']:.0%})
- **Taste (E-Tongue)**: {state['taste_pred']['class']} (confidence: {state['taste_pred']['confidence']:.0%})
- **Vision (Deposit Pattern)**: {state['vision_pred']['class']} (confidence: {state['vision_pred']['confidence']:.0%})

**Correlation Engine Verdict**:
- **Suspected Adulterant**: {state['suspected_adulterant']}
- **Confidence**: {state['correlation_confidence']:.0%}
- **Matched Rules**: {state['matched_rules']}

**Task**:
1. Based on the evidence above, explain why this milk is likely {state['suspected_adulterant']}.
2. Reference the specific evidence from each sensor modality.
3. Assign a food safety risk level: Low, Medium, High, or Critical.
4. Provide a brief recommendation for the next investigation step.

**Format your response as**:
```
RISK LEVEL: [Low/Medium/High/Critical]
REASONING:
[Your explanation here]
RECOMMENDATION:
[Next step]
```
"""

        state["_prompt"] = prompt
        return state

    def call_llm(self, state: FoodSafetyAgentState) -> FoodSafetyAgentState:
        """
        Call Ollama Qwen3 with the prompt.
        """
        prompt = state.get("_prompt", "")

        try:
            print(f"[LLM] Calling Qwen3...")
            response = self.ollama_client.generate(
                model="qwen3",
                prompt=prompt,
                stream=False
            )

            llm_response = response.get("response", "")
            is_fallback = response.get("fallback", False)

            if is_fallback:
                print("[LLM] Using fallback response")
            else:
                print("[LLM] Received response from Qwen3")

        except Exception as e:
            print(f"[LLM ERROR] {e}. Using fallback.")
            llm_response = self._fallback_reasoning(state)

        return {
            **state,
            "llm_response": llm_response
        }

    def _fallback_reasoning(self, state: FoodSafetyAgentState) -> str:
        """
        Fallback reasoning if LLM is unavailable.
        Still produces reasonable output.
        """
        adulterant = state["suspected_adulterant"]
        confidence = state["correlation_confidence"]

        explanations = {
            "Fresh": "The milk shows consistent signals across all modalities (aroma, taste, vision) with no adulterant markers. Clean radial deposit pattern, balanced taste profile, and normal aroma profile are consistent with authentic fresh milk.",
            "Water": "Low confidence signals across all modalities indicate dilution. Weak evaporative deposit, faint aroma, and diluted taste suggest water adulterant present (estimated 5-20% adulteration).",
            "Urea": "High ammonia signal in aroma, high bitterness in taste, and fine crystal pattern in vision are consistent with urea adulteration. Classic triplet signature of urea (3-5% typical adulteration).",
            "Detergent": "Needle-like crystal formation in deposit pattern, harsh taste, and elevated aroma signals indicate detergent/surfactant contamination. Likely from inadequate cleaning of dairy equipment.",
            "Formalin": "Pungent formaldehyde aroma, sour taste, and dense dark deposit are consistent with formalin (preservative) contamination. This presents CRITICAL food safety risk due to toxicity.",
            "H2O2": "Oxidative markers in aroma and center blob in vision deposit suggest hydrogen peroxide use as preservative. Clean taste profile indicates lower adulteration level.",
            "Spoiled": "Sour aroma and taste combined with dark irregular deposit indicate microbial fermentation and spoilage. Likely pathogenic bacteria present; milk is unsafe for consumption."
        }

        risk_levels = {
            "Fresh": "Low",
            "Water": "Medium",
            "Urea": "High",
            "Detergent": "High",
            "Formalin": "Critical",
            "H2O2": "High",
            "Spoiled": "Critical"
        }

        explanation = explanations.get(adulterant, f"The milk shows markers consistent with {adulterant} adulteration.")
        risk = risk_levels.get(adulterant, "Medium")

        fallback_response = f"""RISK LEVEL: {risk}
REASONING:
{explanation}
RECOMMENDATION:
Send to laboratory for confirmatory testing. Do not release for public consumption if risk is High or Critical."""

        return fallback_response

    def parse_response(self, state: FoodSafetyAgentState) -> FoodSafetyAgentState:
        """
        Parse LLM response to extract risk level and reasoning.
        """
        response = state["llm_response"]

        # Extract risk level
        risk_level = "Medium"  # default
        if "CRITICAL" in response.upper():
            risk_level = "Critical"
        elif "HIGH" in response.upper():
            risk_level = "High"
        elif "MEDIUM" in response.upper():
            risk_level = "Medium"
        elif "LOW" in response.upper():
            risk_level = "Low"

        # Extract reasoning (everything after REASONING: until RECOMMENDATION:)
        reasoning = response
        if "REASONING:" in response:
            start = response.find("REASONING:") + len("REASONING:")
            if "RECOMMENDATION:" in response[start:]:
                end = response.find("RECOMMENDATION:", start)
                reasoning = response[start:end].strip()
            else:
                reasoning = response[start:].strip()

        final_verdict = f"{state['suspected_adulterant']} ({risk_level})"

        return {
            **state,
            "risk_level": risk_level,
            "reasoning": reasoning,
            "final_verdict": final_verdict
        }

    def finalize(self, state: FoodSafetyAgentState) -> FoodSafetyAgentState:
        """
        Update investigation record and log execution.
        """
        # Update investigation with final results
        update_investigation(
            investigation_id=state["investigation_id"],
            updates={
                "risk_level": state["risk_level"],
                "reasoning": state["reasoning"]
            },
            db_path=self.db_path
        )

        # Log execution
        execution_id = insert_agent_execution(
            investigation_id=state["investigation_id"],
            agent_name="FoodSafetyAgent",
            input_data={
                "suspected_adulterant": state["suspected_adulterant"],
                "correlation_confidence": state["correlation_confidence"],
                "aroma_pred": state["aroma_pred"],
                "taste_pred": state["taste_pred"],
                "vision_pred": state["vision_pred"]
            },
            output_data={
                "risk_level": state["risk_level"],
                "final_verdict": state["final_verdict"]
            },
            reasoning=state["reasoning"][:200],  # First 200 chars
            execution_time_ms=state.get("execution_time_ms", 0),
            db_path=self.db_path
        )

        return {
            **state,
            "execution_id": execution_id
        }

    def invoke(self, investigation_id: str) -> Dict[str, Any]:
        """
        Run the food safety agent on an investigation.

        1. Fetch investigation & predictions from DB
        2. Execute the LangGraph workflow
        3. Return final state
        """
        # Fetch investigation
        investigation = get_investigation(investigation_id, db_path=self.db_path)
        if not investigation:
            raise ValueError(f"Investigation {investigation_id} not found")

        batch_id = investigation.get("batch_id")

        # Fetch predictions
        aroma = get_aroma_analysis(batch_id, db_path=self.db_path)
        taste = get_taste_analysis(batch_id, db_path=self.db_path)
        vision = get_vision_analysis(batch_id, db_path=self.db_path)

        if not (aroma and taste and vision):
            raise ValueError(f"Incomplete predictions for investigation {investigation_id}")

        # Count matched rules
        rules_query = "SELECT COUNT(*) as count FROM correlation_rules LIMIT 1"
        rules_rows = execute_query(self.db_path, rules_query)
        matched_rules_count = dict_from_row(rules_rows[0]).get("count", 0) if rules_rows else 0

        # Build initial state
        initial_state = FoodSafetyAgentState(
            investigation_id=investigation_id,
            batch_id=batch_id,
            suspected_adulterant=investigation.get("predicted_class", "Unknown"),
            correlation_confidence=investigation.get("confidence", 0.0),
            aroma_pred={
                "class": aroma.get("predicted_class"),
                "confidence": aroma.get("confidence", 0.0)
            },
            taste_pred={
                "class": taste.get("predicted_class"),
                "confidence": taste.get("confidence", 0.0)
            },
            vision_pred={
                "class": vision.get("predicted_class"),
                "confidence": vision.get("confidence", 0.0)
            },
            matched_rules=matched_rules_count,
            llm_response="",
            risk_level="",
            reasoning="",
            final_verdict="",
            execution_id=None,
            execution_time_ms=0.0
        )

        # Execute
        start_time = time.time()
        final_state = self.compiled_graph.invoke(initial_state)
        execution_time_ms = (time.time() - start_time) * 1000

        final_state["execution_time_ms"] = execution_time_ms

        return final_state

print("[OK] FoodSafetyAgent class defined")

## 5. Instantiate Agent

In [ ]:
# Create agent instance
agent = FoodSafetyAgent(db_path=DB_PATH, ollama_client=ollama_client)
print("[OK] FoodSafetyAgent instantiated")

## 6. Get Test Investigation ID

In [ ]:
# Get an investigation from the correlation engine output
query = "SELECT id, batch_id, predicted_class FROM investigations LIMIT 1"
rows = execute_query(DB_PATH, query)
if rows:
    inv_dict = dict_from_row(rows[0])
    test_investigation_id = inv_dict["id"]
    print(f"Test Investigation ID: {test_investigation_id}")
    print(f"  Batch: {inv_dict['batch_id']}")
    print(f"  Predicted: {inv_dict['predicted_class']}")
else:
    print("[ERROR] No investigations found. Run notebooks 00 & 05 first.")

## 7. Run Agent on Test Investigation

In [ ]:
print(f"Running FoodSafetyAgent on investigation {test_investigation_id}...\n")

result = agent.invoke(test_investigation_id)

print("="*60)
print("FOOD SAFETY AGENT RESULT")
print("="*60)
print(f"\nInvestigation ID: {result['investigation_id']}")
print(f"\nEvidence:")
print(f"  Aroma: {result['aroma_pred']['class']} ({result['aroma_pred']['confidence']:.0%})")
print(f"  Taste: {result['taste_pred']['class']} ({result['taste_pred']['confidence']:.0%})")
print(f"  Vision: {result['vision_pred']['class']} ({result['vision_pred']['confidence']:.0%})")
print(f"\nCorrelation Agent Verdict:")
print(f"  Suspected: {result['suspected_adulterant']}")
print(f"  Confidence: {result['correlation_confidence']:.0%}")
print(f"\nLLM Risk Assessment:")
print(f"  Risk Level: {result['risk_level']}")
print(f"\nLLM Reasoning (first 300 chars):")
print(f"  {result['reasoning'][:300]}...")
print(f"\n{'='*60}")
print(f"FINAL VERDICT: {result['final_verdict']}")
print(f"Execution Time: {result['execution_time_ms']:.1f}ms")
print(f"{'='*60}")

## 8. Test: Run Multiple Investigations

In [ ]:
# Get multiple investigations and run agent on each
query = "SELECT id FROM investigations LIMIT 5"
rows = execute_query(DB_PATH, query)
investigation_ids = [dict_from_row(row)["id"] for row in rows]

print(f"Testing agent on {len(investigation_ids)} investigations...\n")

results_summary = []
for i, inv_id in enumerate(investigation_ids, 1):
    try:
        result = agent.invoke(inv_id)
        results_summary.append({
            "investigation_id": inv_id,
            "suspected": result["suspected_adulterant"],
            "risk_level": result["risk_level"],
            "exec_time_ms": result["execution_time_ms"]
        })
        print(f"{i}. {inv_id}")
        print(f"   Suspected: {result['suspected_adulterant']}")
        print(f"   Risk: {result['risk_level']}")
        print(f"   Time: {result['execution_time_ms']:.1f}ms")
    except Exception as e:
        print(f"{i}. {inv_id} → ERROR: {e}")

print(f"\n[OK] Processed {len(results_summary)} investigations")

## 9. Verify Agent Execution Logging

In [ ]:
# Check agent_execution table for FoodSafetyAgent entries
query = "SELECT * FROM agent_execution WHERE agent_name = 'FoodSafetyAgent' ORDER BY created_at DESC LIMIT 5"
rows = execute_query(DB_PATH, query)

print(f"FoodSafetyAgent executions logged: {len(rows)}\n")
for i, row in enumerate(rows, 1):
    exec_dict = dict_from_row(row)
    print(f"{i}. ID: {exec_dict['id']}")
    print(f"   Investigation: {exec_dict['investigation_id']}")
    print(f"   Agent: {exec_dict['agent_name']}")
    output_data = json.loads(exec_dict.get("output_data", "{}"))
    print(f"   Verdict: {output_data.get('final_verdict', 'N/A')}")
    print()

## 10. Check Ollama Integration

In [ ]:
# Summary of LLM integration
print("LLM Integration Summary:")
print(f"  Ollama Available: {is_available}")
print(f"  Model: Qwen3")
print(f"  Fallback: Enabled (uses template-based reasoning if Qwen3 unavailable)")
print(f"\nFoodSafetyAgent performs:")
print(f"  1. Prompt building with all evidence")
print(f"  2. LLM call to Qwen3 (or fallback)")
print(f"  3. Response parsing for risk level & reasoning")
print(f"  4. Investigation update & execution logging")

## ✅ Food Safety Reasoning Agent Complete!

**Summary**:
- ✓ LangGraph FoodSafetyAgent implemented
- ✓ 4-node workflow: build_prompt → call_llm → parse_response → finalize
- ✓ Ollama Qwen3 integration (with fallback)
- ✓ Risk level extraction & reasoning parsing
- ✓ Investigation updates with final verdict
- ✓ Agent execution logging to DB
- ✓ Tested on synthetic data

**Next Steps**:
1. Run `07_passport_certificate.ipynb` for report & passport generation
2. Run `08_gradio_demo.ipynb` for full end-to-end UI demo